# VectorStore Retriever — 벡터 기반 문서 검색기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **VectorStoreRetriever**를 `as_retriever()`로 생성하는 방법 이해하기
- Similarity, MMR, Score Threshold 세 가지 검색 방식의 차이 파악하기
- `ConfigurableField`로 런타임에 검색 파라미터를 동적으로 바꾸기

## 사전 지식

- 6-4 VectorStore(벡터 저장소)에서 FAISS 벡터스토어를 만들어본 경험
- 임베딩(Embedding)이 텍스트를 숫자 벡터로 변환한다는 개념

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> E[쿼리 임베딩]:::process
    E --> V[(FAISS<br/>벡터스토어)]:::storage
    V --> S{검색 방식}:::process
    S -- Similarity --> R1[유사도<br/>Top-K]:::output
    S -- MMR --> R2[다양성<br/>고려]:::output
    S -- Threshold --> R3[임계값<br/>필터링]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

`VectorStoreRetriever`는 벡터 데이터베이스를 활용하여 의미적으로 유사한 문서를 검색하는 가장 기본적인 Retriever(검색기)입니다.

## 주요 특징

| 특징 | 설명 |
|------|------|
| 의미 기반 검색 | 임베딩 벡터 간 유사도(코사인 유사도)로 검색 |
| 다양한 검색 방식 | Similarity, MMR, Score Threshold 지원 |
| 동적 설정 | `ConfigurableField`로 런타임 파라미터 조정 |
| 통합 인터페이스 | 모든 벡터 DB에서 동일한 API 사용 |

## 환경 설정

In [1]:
import os
from dotenv import load_dotenv

# API 키 정보 로드

# 아래에 코드를 작성하세요
load_dotenv()


True

## 1. VectorStore 생성

먼저 문서를 로드하고 벡터 데이터베이스를 생성합니다.

In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# ---------------------------------------------------
# 문서를 로드하여 VectorStore 생성을 위한 원본 텍스트 준비
# ---------------------------------------------------

# ============================================================
# TODO: 문서를 로드하고 분할 수와 전체 길이를 출력하세요
# 힌트: TextLoader로 파일을 읽은 뒤 .load() 호출
# 예상 결과: 로드된 문서 수와 문자 수 출력
# ============================================================

# 1단계: 문서 로드
# TextLoader: 텍스트 파일을 Document 객체로 변환하는 로더
documents = TextLoader("./data/ai-story.txt").load()


In [6]:
# ---------------------------------------------------
# 문서를 작은 청크로 분할하여 검색 단위 생성
# ---------------------------------------------------

# ============================================================
# TODO: RecursiveCharacterTextSplitter로 문서를 분할하세요
# 힌트: chunk_size=500, chunk_overlap=50 설정
# 예상 결과: 분할된 청크 수와 첫 번째 청크 내용 출력
# ============================================================

# 2단계: 문서 분할
# RecursiveCharacterTextSplitter: 단락, 줄바꿈, 공백 순서로 재귀 분할
# chunk_overlap: 인접 청크 간 중복 문자 수 (문맥 연결 유지)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(documents)



In [7]:
# 3단계: OpenAI 임베딩 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# 4단계: FAISS 벡터 데이터베이스 생성
vectorstore = FAISS.from_documents(split_docs, embeddings)

# 아래에 코드를 작성하세요
vectorstore.index.ntotal


19

## 2. VectorStoreRetriever 생성

`as_retriever()` 메서드는 VectorStore를 Retriever로 변환합니다.

### 주요 파라미터

- `search_type`: 검색 방식 선택
  - `"similarity"`: 유사도 기반 검색 (기본값)
  - `"mmr"`: Maximal Marginal Relevance - 다양성 고려
  - `"similarity_score_threshold"`: 임계값 이상의 문서만 반환

- `search_kwargs`: 검색 옵션 설정
  - `k`: 반환할 문서 수 (기본값: 4)
  - `score_threshold`: 최소 유사도 점수 (0~1)
  - `fetch_k`: MMR의 후보 문서 수 (기본값: 20)
  - `lambda_mult`: MMR 다양성 파라미터 (0~1, 기본값: 0.5)

In [9]:
# ---------------------------------------------------
# VectorStore를 Retriever 인터페이스로 변환
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 기본 retriever를 생성하고 설정을 확인하세요
# 힌트: vectorstore.as_retriever() — 기본값은 similarity, k=4
# 예상 결과: search_type과 search_kwargs 출력
# ============================================================

# as_retriever(): VectorStore → Retriever 변환
# search_type 기본값: "similarity" (코사인 유사도)
# search_kwargs 기본값: {"k": 4}
retriever = vectorstore.as_retriever()

## 3. 기본 검색 - Similarity Search

가장 유사한 벡터를 찾아 관련 문서를 반환합니다.

In [13]:
# 쿼리: Transformer 모델에 대한 질문
query = "Transformer 모델은 어떤 메커니즘을 기반으로 하나요?"

# invoke() 메서드로 문서 검색
docs = retriever.invoke(query)

print(len(docs))
print(docs[3].page_content)


4
Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이 포함되며, 다양한 언어에 대한 지원도 제공합니다. 라이브러리는 사전 훈련된 모델을 포함하고 있어, 사용자가 복잡한 모델을 처음부터 훈련시키지 않고도, 빠르게 고성능의 NLP 시스템을 구축할 수 있게 합니다.

Hugging Face는 또한 'Model Hub'를 운영하고 있습니다. 이는 수천 개의 사전 훈련된 모델과 데이터셋을 호스팅하는 플랫폼으로, 연구자와 개발자가 자신의 모델을 공유하고 다른 사람들의 모델을 쉽게 찾아 사용할 수 있게 합니다. Model Hub를 통해 사용자는 특정 NLP 작업에 최적화된 모델을 검색하고, 직접 훈련시킨 모델을 커뮤니티와 공유할 수 있습니다.


## 4. MMR (Maximal Marginal Relevance) 검색

MMR은 관련성과 다양성을 모두 고려하는 검색 방식입니다.

### 작동 원리

1. 먼저 `fetch_k`개의 후보 문서를 검색 (유사도 기준)
2. 후보 문서 중에서 이미 선택된 문서와 **다양성**을 고려하여 `k`개 선택
3. `lambda_mult` 파라미터로 관련성과 다양성의 균형 조절
   - `lambda_mult = 1.0`: 오직 관련성만 고려 (similarity search와 동일)
   - `lambda_mult = 0.0`: 오직 다양성만 고려
   - `lambda_mult = 0.5`: 관련성과 다양성 균형 (기본값)

In [15]:
# MMR retriever 생성
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.6
    }
)

# 동일한 쿼리로 검색
mmr_docs = mmr_retriever.invoke(query)
len(mmr_docs)
print(mmr_docs[0].page_content)


"Attention Is All You Need" 논문은 변환기 아키텍처를 기반으로 한 '멀티헤드 어텐션(Multi-Head Attention)' 기법을 도입했습니다. 이는 여러 개의 어텐션 메커니즘을 병렬로 사용하여, 서로 다른 표현 공간에서 정보를 동시에 처리함으로써 모델의 성능을 향상시킵니다. 또한, 논문은 전통적인 RNN이나 CNN에 의존하지 않음으로써 발생하는 병렬 처리의 이점을 강조하며, 이를 통해 훨씬 더 빠른 학습 속도와 효율적인 메모리 사용을 실현합니다.

변환기 모델의 이러한 혁신적인 접근 방식은 NLP를 비롯한 여러 분야에서 큰 변화를 가져왔습니다. BERT, GPT, RoBERTa 등의 후속 모델들은 모두 "Attention Is All You Need" 논문에서 제시된 변환기 아키텍처를 기반으로 하며, 이들 모델은 텍스트 분류, 요약, 번역, 질문 응답 시스템 등 다양한 작업에서 인상적인 성능을 보여줍니다.


### MMR vs Similarity 비교

동일한 쿼리에 대해 두 검색 방식을 비교해봅시다.

In [ ]:
# ---------------------------------------------------
# 동일 쿼리로 Similarity와 MMR 결과를 나란히 비교
# ---------------------------------------------------

# ============================================================
# TODO: 동일 쿼리에 sim_retriever와 mmr_retriever를 모두 실행하고 결과를 비교하세요
# 힌트: 두 retriever에 동일한 compare_query를 invoke()
# 예상 결과: 두 방식의 상위 문서 목록 — 중복 여부 차이 관찰
# ============================================================

# 비교 쿼리

# 1. Similarity search

# 2. MMR search


## 5. Score Threshold 검색

유사도 점수가 특정 임계값 이상인 문서만 반환합니다.

### 언제 사용하면 좋을까요?

- 관련성이 낮은 문서를 필터링하고 싶을 때
- 정확도가 중요한 애플리케이션에서
- "관련 문서가 없으면 아무것도 반환하지 않음" 동작이 필요할 때

In [17]:
# Score threshold retriever 생성
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.5
    }
)
# 테스트 쿼리 1: 관련성 높은 쿼리
high_relevance_query = "Word2Vec은 무엇인가요?"
docs_high = threshold_retriever.invoke(high_relevance_query)
print(len(docs_high))
print(docs_high[0].page_content)


# 테스트 쿼리 2: 관련성 낮은 쿼리

# 아래에 코드를 작성하세요


2
Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 단어 임베딩 기법 중 하나입니다. 2013년 Google의 연구팀에 의해 개발되었으며, 단어를 벡터 공간에 매핑함으로써 컴퓨터가 단어의 의미를 수치적으로 이해할 수 있게 합니다. Word2Vec의 핵심 아이디어는 비슷한 맥락에서 사용되는 단어들은 비슷한 벡터를 가진다는 것입니다. 이를 통해 단어 간의 의미적 유사성과 관계를 벡터 연산을 통해 표현할 수 있습니다.

Word2Vec은 크게 두 가지 모델 아키텍처로 구성됩니다: Continuous Bag-of-Words (CBOW)와 Skip-Gram입니다. CBOW 모델은 주변 단어(맥락)를 기반으로 특정 단어를 예측하는 반면, Skip-Gram 모델은 하나의 단어로부터 주변 단어들을 예측합니다. 두 모델 모두 딥러닝이 아닌, 단순화된 신경망 구조를 사용하여 대규모 텍스트 데이터에서 학습할 수 있으며, 매우 효율적입니다.


## 6. Top-k 설정

반환할 문서 개수를 동적으로 조절할 수 있습니다.

In [18]:
# ---------------------------------------------------
# k 값에 따른 검색 결과 수 변화 실험
# ---------------------------------------------------

# ============================================================
# TODO: k=1, 3, 5 각각으로 retriever를 생성하고 결과 수를 확인하세요
# 힌트: search_kwargs={"k": k} 로 k를 변수로 전달
# 예상 결과: k 값이 커질수록 반환 문서 수가 늘어남
# ============================================================

# 다양한 k 값으로 검색

# k=1, 3, 5 순서로 실험 — k가 클수록 더 많은 문서 검색
test_query = "딥러닝과 머신러닝의 차이는 무엇인가?"

for k in [1, 3, 5]:
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    docs_k = retriever_k.invoke(test_query)

    print(docs_k)
    print("=" * 100)


[Document(id='71bbde13-1752-4101-ad93-85f19c45574c', metadata={'source': './data/ai-story.txt'}, page_content='NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 파악하고, 문맥을 분석하여 의미를 정확하게 이해할 수 있어야 합니다. 이를 위해 NLP는 통계학, 기계 학습, 딥 러닝과 같은 다양한 수학적 및 계산적 방법론을 활용합니다.\n\n최근 몇 년 동안, 딥 러닝 기반의 NLP 모델, 특히 변환기(Transformer) 아키텍처와 같은 신경망 모델의 발전으로 인해, 자연어 이해와 생성 분야에서 놀라운 진보가 이루어졌습니다. 이러한 모델들은 대규모의 언어 데이터에서 학습하여, 문장의 문맥을 효과적으로 파악하고, 보다 정교한 언어 이해 및 생성 능력을 제공합니다.')]
[Document(id='71bbde13-1752-4101-ad93-85f19c45574c', metadata={'source': './data/ai-story.txt'}, page_content='NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 파악하고, 문맥을 분석하여 의미를 정확하게 이해할 수 있어야 합니다. 이를 위해 NLP는 통계학, 기계 학습, 딥 러닝과 같은 다양한 수학적 및 계산적 방법론을 활용합니다.\n\n최근 몇 년 동안, 딥 러닝 기반의 NLP 모델, 특히 변환기(Transformer) 아키텍처와 같은 신경망 모델의 발전으로 인해, 자연어 이해와 생성 분야에서 

## 7. 동적 설정 (ConfigurableField)

`ConfigurableField`를 사용하면 Retriever 생성 후에도 검색 파라미터를 동적으로 변경할 수 있습니다.

### 언제 사용하면 좋을까요?

- 사용자별로 다른 검색 설정을 적용할 때
- A/B 테스트로 최적의 검색 파라미터를 찾을 때
- 런타임에 검색 전략을 변경해야 할 때

> **실무 팁**: `ConfigurableField`는 프로덕션 환경에서 특히 유용합니다. 사용자 등급별로 k 값을 다르게 적용하거나, 특정 사용자에게만 MMR 검색을 제공하는 등 유연한 서비스가 가능해요.

In [19]:
from langchain_core.runnables import ConfigurableField

# 동적 설정이 가능한 retriever 생성
configurable_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
).configurable_fields(
    search_type=ConfigurableField(
        id="search_type",
        name="검색 타입",
        description="검색 방식 지정: similarity, mmr, similarity_score_threshold 중 하나"
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs",
        name="Search Kwargs",
        description="검색 파라미터: k, fetch_k, lambda_mult, score_threshold 등",
    ),
)



# 아래에 코드를 작성하세요


In [21]:
# ---------------------------------------------------
# ConfigurableField로 k=4 설정을 런타임에 적용
# ---------------------------------------------------

# ============================================================
# TODO: config={"configurable": {"search_kwargs": {"k": 4}}} 로 검색하세요
# 힌트: configurable_retriever.invoke(query, config=config1)
# 예상 결과: 4개 문서 반환
# ============================================================

# 예제 1: k=4로 검색
# configurable 딕셔너리 — 런타임에 설정을 덮어씀
from langchain_core.runnables import RunnableConfig


config1: RunnableConfig = {"configurable": {"search_kwargs": {"k": 4}}}

test_query = "Huggingface는 무엇은 제공하나요?"
docs_config1 = configurable_retriever.invoke(test_query, config=config1)

print(len(docs_config1))


4


In [30]:
# ---------------------------------------------------
# 런타임에 검색 방식을 Score Threshold로 교체
# ---------------------------------------------------

# ============================================================
# TODO: search_type을 "similarity_score_threshold"로 바꾸고 threshold=0.75로 검색하세요
# 힌트: config2의 "search_type" 키에 새 방식을 문자열로 전달
# 예상 결과: 유사도 0.75 이상의 문서만 반환 (없으면 빈 리스트)
# ============================================================

# 예제 2: Score threshold 방식으로 변경
# search_type을 런타임에 교체 — 재생성 없이 검색 전략을 바꿀 수 있음
config2: RunnableConfig = {"configurable": {"search_kwargs": {"score_threshold": 0.4}, "search_type": "similarity_score_threshold" }}

docs_config2 = configurable_retriever.invoke(test_query, config=config2)

print(len(docs_config2))
print(docs_config2)

2
[Document(id='4894b1bb-48c9-4aa5-bab5-c8404fc34196', metadata={'source': './data/ai-story.txt'}, page_content='Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 대해 적극적으로 논의하고, 모델 배포 시 발생할 수 있는 윤리적 문제를 해결하기 위한 가이드라인을 제공하려 노력합니다.\n\n종합적으로 볼 때, Hugging Face는 NLP 및 AI 분야에서 중요한 자원을 제공하는 회사로, 오픈 소스 기여와 커뮤니티 구축을 통해 AI 기술의 발전과 보급에 기여하고 있습니다. 이를 통해 연구자, 개발자, 기업 등 다양한 사용자가 최신 AI 기술을 쉽게 접근하고 활용할 수 있는 환경을 조성하고 있습니다.\n\nAttention is all you need'), Document(id='33f12704-3af0-49eb-8f47-d029f60df351', metadata={'source': './data/ai-story.txt'}, page_content="HuggingFace\n\nHugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transformers' 라이브러리를 통해 큰 인기를 얻었습니다. Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다.")]


In [26]:
# 예제 3: MMR 방식으로 변경

# search_type="mmr",
#     search_kwargs={
#         "k": 3,
#         "fetch_k": 10,
#         "lambda_mult": 0.6
#     }

config3: RunnableConfig = {"configurable": {"search_kwargs": {"k": 3, "fetch_k": 10, "lambda_mult": 0.6}, "search_type": "mmr" }}
docs_config3 = configurable_retriever.invoke(test_query, config=config3)

print(len(docs_config3))
print(docs_config3)

3
[Document(id='4894b1bb-48c9-4aa5-bab5-c8404fc34196', metadata={'source': './data/ai-story.txt'}, page_content='Hugging Face는 또한 AI 연구 커뮤니티에 대한 기여를 넘어, 윤리적 AI 개발과 모델의 공정성 및 투명성에 대한 중요성을 강조합니다. 회사는 AI 기술의 사회적 영향에 대해 적극적으로 논의하고, 모델 배포 시 발생할 수 있는 윤리적 문제를 해결하기 위한 가이드라인을 제공하려 노력합니다.\n\n종합적으로 볼 때, Hugging Face는 NLP 및 AI 분야에서 중요한 자원을 제공하는 회사로, 오픈 소스 기여와 커뮤니티 구축을 통해 AI 기술의 발전과 보급에 기여하고 있습니다. 이를 통해 연구자, 개발자, 기업 등 다양한 사용자가 최신 AI 기술을 쉽게 접근하고 활용할 수 있는 환경을 조성하고 있습니다.\n\nAttention is all you need'), Document(id='33f12704-3af0-49eb-8f47-d029f60df351', metadata={'source': './data/ai-story.txt'}, page_content="HuggingFace\n\nHugging Face는 자연어 처리(NLP) 분야에서 선도적인 역할을 하는 기술 회사로, 인공 지능(AI) 연구와 응용 프로그램 개발을 위한 다양한 오픈 소스 도구와 라이브러리를 제공합니다. 2016년에 설립된 이 회사는 특히 'Transformers' 라이브러리를 통해 큰 인기를 얻었습니다. Transformers 라이브러리는 BERT, GPT, RoBERTa, T5와 같은 최신의 변환기(Transformer) 기반 모델을 쉽게 사용할 수 있게 해주며, Python 프로그래밍 언어로 작성되어 있습니다. 이 라이브러리의 주요 목적은 최신의 NLP 기술을 쉽고 효율적으로 접근할 수 있도록 하는 것입니다."), Document(id='d140a0c9-1db5-4551-8952-3

## 8. Query와 Document에 다른 임베딩 모델 사용하기

일부 임베딩 모델(예: Upstage Solar)은 쿼리용과 문서용 임베딩을 분리하여 제공합니다.

### 왜 분리할까요?

- **쿼리**: 짧고 검색 의도가 명확한 텍스트
- **문서**: 길고 정보가 풍부한 텍스트
- 각각에 최적화된 임베딩 모델을 사용하면 검색 품질이 향상됩니다.

In [ ]:
# 주의: 이 예제를 실행하려면 UPSTAGE_API_KEY가 필요합니다.
# .env 파일에 UPSTAGE_API_KEY를 추가하세요.


from langchain_upstage import UpstageEmbeddings

# 1단계: 문서용 임베딩으로 벡터스토어 생성

# 문서 재로드 및 분할

# 문서용 임베딩으로 벡터스토어 생성

# 아래에 코드를 작성하세요


In [ ]:
# 2단계: 쿼리용 임베딩으로 검색

# 쿼리를 쿼리용 임베딩으로 변환

# 벡터 유사도 검색

# 아래에 코드를 작성하세요


## 9. 검색 전략 비교 요약

### 검색 방식별 특징

| 검색 방식 | 장점 | 단점 | 추천 사용 시나리오 |
|---------|------|------|------------------|
| **Similarity** | 빠르고 간단, 높은 관련성 | 중복 결과 가능 | 일반적인 검색, 빠른 응답 필요 시 |
| **MMR** | 다양성 확보, 중복 감소 | 약간 느림 | 다양한 관점 필요, 탐색적 검색 |
| **Score Threshold** | 품질 보장, 무의미한 결과 방지 | 결과 없을 수 있음 | 정확도 중요, 낮은 관련성 필터링 |

### 파라미터 설정 가이드

- **k**: 일반적으로 3~5개가 적절 (너무 많으면 노이즈 증가)
- **fetch_k**: k의 2~4배 정도 (MMR에서)
- **lambda_mult**: 
  - 0.7~0.9: 관련성 우선 (정확도 중요 시)
  - 0.3~0.5: 다양성 우선 (탐색적 검색 시)
- **score_threshold**: 
  - 0.8~1.0: 매우 엄격 (고품질 필요 시)
  - 0.6~0.8: 적당한 필터링
  - 0.5 이하: 너무 느슨함

> **실무 팁**: 실제 서비스에서는 Similarity(속도)로 시작하고, 응답 품질 이슈가 생기면 MMR이나 Threshold를 추가하는 방식으로 단계적으로 개선하세요. 처음부터 복잡하게 만들 필요 없어요.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| 검색 방식 | 언제 사용? | 핵심 파라미터 |
|----------|-----------|--------------|
| **Similarity** | 빠른 기본 검색 | `k` (반환 문서 수) |
| **MMR** | 다양성이 필요할 때 | `lambda_mult`, `fetch_k` |
| **Score Threshold** | 품질 기준 이상만 반환 | `score_threshold` (0~1) |
| **ConfigurableField** | 런타임 설정 변경 | `configurable` dict |

### 다음 노트북 예고

**02-ContextualCompressionRetriever**: 검색된 문서에서 관련 내용만 추출해 노이즈를 줄이는 방법을 배워요. 검색 품질과 LLM 비용을 동시에 개선하는 핵심 기법입니다.